In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

from flask import Flask, render_template_string, request, jsonify
import pandas as pd
import numpy as np
import joblib
import threading
import time
from werkzeug.serving import make_server


# ============================================================
# 2. CREATE FLASK APP
# ============================================================

app = Flask(__name__)


# ============================================================
# 3. LOAD SAVED MODEL AND PREPROCESSING OBJECTS
# ============================================================

model = joblib.load("model/final_xgb_model.pkl")
imputer = joblib.load("model/imputer.pkl")
label_encoder = joblib.load("model/label_encoder.pkl")


# Columns that required imputation during training
impute_cols = [
    "workclass",
    "occupation",
    "native-country"
]

# adding median value of fnlwgt from trained data. Hardcored.
fnlwgt_median = 178356.0

# ============================================================
# 4. FEATURE COLUMNS
# ============================================================

feature_columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "hours-per-week",
    "native-country",
    "capital-gain",
    "capital-loss"
]


# ============================================================
# 5. HTML FORM
# ============================================================

HTML_FORM = """
<!DOCTYPE html>
<html>

<head>

<meta charset="UTF-8">

<title>Census Income Prediction</title>

<style>

body {
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #f4f6f8;
    margin: 0;
    padding: 0 0 40px 0;
}

.container {
    max-width: 850px;
    margin: 40px auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.08);
    padding: 32px 40px;
}

h1 {
    text-align: center;
    color: #2c3e50;
    margin-bottom: 4px;
}

.subtitle {
    text-align: center;
    color: #7f8c8d;
    margin-top: 0;
    margin-bottom: 28px;
    font-size: 14px;
}

form {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px 20px;
}

.field {
    display: flex;
    flex-direction: column;
}

label {
    font-size: 13px;
    color: #34495e;
    margin-bottom: 4px;
    font-weight: 600;
}

input,
select {
    padding: 9px 10px;
    border: 1px solid #cfd8dc;
    border-radius: 6px;
    font-size: 14px;
}

input:focus,
select:focus {
    outline: none;
    border-color: #3498db;
}

.submit-row {
    grid-column: 1 / -1;
    margin-top: 8px;
}

button {
    width: 100%;
    padding: 12px;
    background: #2980b9;
    color: white;
    border: none;
    border-radius: 6px;
    font-size: 15px;
    font-weight: 600;
    cursor: pointer;
}

button:hover {
    background: #21618c;
}

.result {
    margin-top: 26px;
    padding: 18px;
    border-radius: 8px;
    text-align: center;
    font-size: 18px;
    font-weight: 600;
}

.result.high {
    background: #eafaf1;
    color: #1e8449;
    border: 1px solid #a9dfbf;
}

.result.low {
    background: #fdecea;
    color: #c0392b;
    border: 1px solid #f5b7b1;
}

.prob {
    display: block;
    font-size: 13px;
    font-weight: 400;
    margin-top: 6px;
    color: #555;
}

.error {
    margin-top: 20px;
    padding: 14px;
    background: #fdecea;
    border: 1px solid #f5b7b1;
    color: #c0392b;
    border-radius: 6px;
    font-size: 14px;
}

</style>

</head>

<body>

<div class="container">

<h1>Census Income Predictor</h1>

<p class="subtitle">
Tuned XGBoost model for income classification
</p>


<form action="/predict" method="POST">


<!-- AGE -->

<div class="field">
<label>Age</label>

<input
type="number"
name="age"
required
value="{{ form_values.get('age', '') if form_values else '' }}"
>

</div>


<!-- WORKCLASS -->

<div class="field">
<label>Workclass</label>

<select name="workclass" required>

<option value="">Select</option>
<option>Private</option>
<option>Self-emp-not-inc</option>
<option>Self-emp-inc</option>
<option>Federal-gov</option>
<option>Local-gov</option>
<option>State-gov</option>
<option>Without-pay</option>
<option>Never-worked</option>

</select>

</div>


<!-- FNLWGT -->

<div class="field">
<label>fnlwgt</label>

<input
type="number"
name="fnlwgt"
step="any"
value="{{ form_values.get('fnlwgt', '') if form_values else '' }}"
>

<small style="color:#95a5a6;">
Leave blank if you dont know.
</small>

</div>


<!-- EDUCATION -->

<div class="field">
<label>Education</label>

<select name="education" required>

<option value="">Select</option>
<option>Bachelors</option>
<option>Some-college</option>
<option>11th</option>
<option>HS-grad</option>
<option>Prof-school</option>
<option>Assoc-acdm</option>
<option>Assoc-voc</option>
<option>9th</option>
<option>7th-8th</option>
<option>12th</option>
<option>Masters</option>
<option>1st-4th</option>
<option>10th</option>
<option>Doctorate</option>
<option>5th-6th</option>
<option>Preschool</option>

</select>

</div>


<!-- EDUCATION NUMBER -->

<div class="field">
<label>Education Number</label>

<input
type="number"
name="education-num"
required
value="{{ form_values.get('education-num', '') if form_values else '' }}"
>

</div>


<!-- MARITAL STATUS -->

<div class="field">
<label>Marital Status</label>

<select name="marital-status" required>

<option value="">Select</option>
<option>Married-civ-spouse</option>
<option>Divorced</option>
<option>Never-married</option>
<option>Separated</option>
<option>Widowed</option>
<option>Married-spouse-absent</option>
<option>Married-AF-spouse</option>

</select>

</div>


<!-- OCCUPATION -->

<div class="field">
<label>Occupation</label>

<select name="occupation" required>

<option value="">Select</option>
<option>Tech-support</option>
<option>Craft-repair</option>
<option>Other-service</option>
<option>Sales</option>
<option>Exec-managerial</option>
<option>Prof-specialty</option>
<option>Handlers-cleaners</option>
<option>Machine-op-inspct</option>
<option>Adm-clerical</option>
<option>Farming-fishing</option>
<option>Transport-moving</option>
<option>Priv-house-serv</option>
<option>Protective-serv</option>
<option>Armed-Forces</option>

</select>

</div>


<!-- RELATIONSHIP -->

<div class="field">
<label>Relationship</label>

<select name="relationship" required>

<option value="">Select</option>
<option>Wife</option>
<option>Own-child</option>
<option>Husband</option>
<option>Not-in-family</option>
<option>Other-relative</option>
<option>Unmarried</option>

</select>

</div>


<!-- RACE -->

<div class="field">
<label>Race</label>

<select name="race" required>

<option value="">Select</option>
<option>White</option>
<option>Asian-Pac-Islander</option>
<option>Amer-Indian-Eskimo</option>
<option>Other</option>
<option>Black</option>

</select>

</div>


<!-- SEX -->

<div class="field">
<label>Sex</label>

<select name="sex" required>

<option value="">Select</option>
<option>Male</option>
<option>Female</option>

</select>

</div>


<!-- HOURS PER WEEK -->

<div class="field">
<label>Hours per Week</label>

<input
type="number"
name="hours-per-week"
required
value="{{ form_values.get('hours-per-week', '') if form_values else '' }}"
>

</div>


<!-- NATIVE COUNTRY -->

<div class="field">
<label>Native Country</label>

<select name="native-country" required>

<option value="">Select</option>
<option>United-States</option>
<option>Cambodia</option>
<option>England</option>
<option>Puerto-Rico</option>
<option>Canada</option>
<option>Germany</option>
<option>Outlying-US(Guam-USVI-etc)</option>
<option>India</option>
<option>Japan</option>
<option>Greece</option>
<option>South</option>
<option>China</option>
<option>Cuba</option>
<option>Iran</option>
<option>Honduras</option>
<option>Philippines</option>
<option>Italy</option>
<option>Poland</option>
<option>Jamaica</option>
<option>Vietnam</option>
<option>Mexico</option>
<option>Portugal</option>
<option>Ireland</option>
<option>France</option>
<option>Dominican-Republic</option>
<option>Laos</option>
<option>Ecuador</option>
<option>Taiwan</option>
<option>Haiti</option>
<option>Columbia</option>
<option>Hungary</option>
<option>Guatemala</option>
<option>Nicaragua</option>
<option>Scotland</option>
<option>Thailand</option>
<option>Yugoslavia</option>
<option>El-Salvador</option>
<option>Trinadad&Tobago</option>
<option>Peru</option>
<option>Hong</option>
<option>Holand-Netherlands</option>

</select>

</div>


<!-- CAPITAL GAIN -->

<div class="field">
<label>Capital Gain</label>

<input
type="number"
name="capital-gain"
step="any"
required
value="{{ form_values.get('capital-gain', '') if form_values else '' }}"
>

</div>


<!-- CAPITAL LOSS -->

<div class="field">
<label>Capital Loss</label>

<input
type="number"
name="capital-loss"
step="any"
required
value="{{ form_values.get('capital-loss', '') if form_values else '' }}"
>

</div>


<div class="submit-row">

<button type="submit">
Predict Income
</button>

</div>

</form>


{% if result %}

<div class="result {{ 'high' if result.prediction == 1 else 'low' }}">

Predicted Income: {{ result.label }}

<span class="prob">
Probability of income >50K: {{ result.probability }}%
</span>

</div>

{% endif %}


{% if error %}

<div class="error">

Error: {{ error }}

</div>

{% endif %}


</div>

</body>

</html>
"""


# ============================================================
# 6. PREPARE USER INPUT
# ============================================================

def prepare_input(form_data):

    row = {}

    numerical_cols = [
        "age",
        "fnlwgt",
        "education-num",
        "hours-per-week",
        "capital-gain",
        "capital-loss"
    ]

    for col in feature_columns:

        value = form_data.get(col)

        if col == "fnlwgt" and (value is None or value == ""):
            row[col] = fnlwgt_median

        elif col in numerical_cols:
            row[col] = float(value)

        else:
            row[col] = value

    df_row = pd.DataFrame(
        [row],
        columns=feature_columns
    )


    # Create engineered feature
    df_row["capital-net"] = (
        df_row["capital-gain"] -
        df_row["capital-loss"]
    )


    # Remove original columns
    df_row = df_row.drop(
        ["capital-gain", "capital-loss"],
        axis=1
    )


    # Apply the same imputer used during training
    df_row[impute_cols] = imputer.transform(
        df_row[impute_cols]
    )


    return df_row


# ============================================================
# 7. HOME ROUTE
# ============================================================

@app.route("/", methods=["GET"])

def home():

    return render_template_string(
        HTML_FORM,
        form_values={}
    )


# ============================================================
# 8. PREDICTION ROUTE
# ============================================================

@app.route("/predict", methods=["POST"])

def predict():

    try:

        input_df = prepare_input(request.form)


        prediction_encoded = int(
            model.predict(input_df)[0]
        )


        probability = float(
            model.predict_proba(input_df)[0][1]
        )


        prediction_label = label_encoder.inverse_transform(
            [prediction_encoded]
        )[0]


        result = {

            "prediction": prediction_encoded,

            "label": prediction_label,

            "probability": round(
                probability * 100,
                2
            )
        }


        return render_template_string(
            HTML_FORM,
            result=result,
            form_values=request.form
        )


    except Exception as e:

        return render_template_string(
            HTML_FORM,
            error=str(e),
            form_values=request.form
        )


# ============================================================
# 9. API ROUTE
# ============================================================

@app.route("/api/predict", methods=["POST"])

def api_predict():

    try:

        data = request.get_json(force=True)

        input_df = prepare_input(data)


        prediction_encoded = int(
            model.predict(input_df)[0]
        )


        probability = float(
            model.predict_proba(input_df)[0][1]
        )


        prediction_label = label_encoder.inverse_transform(
            [prediction_encoded]
        )[0]


        return jsonify({

            "prediction": prediction_encoded,

            "label": prediction_label,

            "probability": round(
                probability * 100,
                2
            )

        })


    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 400


# ============================================================
# 10. START FLASK SERVER IN BACKGROUND THREAD
# ============================================================

class ServerThread(threading.Thread):

    def __init__(
        self,
        app,
        host="127.0.0.1",
        port=5000
    ):

        super().__init__()

        self.server = make_server(
            host,
            port,
            app
        )

        self.ctx = app.app_context()
        self.ctx.push()


    def run(self):

        print("Flask server running...")
        print("Open http://localhost:5000")

        self.server.serve_forever()


    def shutdown(self):

        self.server.shutdown()


# ============================================================
# 11. RUN SERVER
# ============================================================

server_thread = ServerThread(
    app,
    port=5000
)

server_thread.start()

time.sleep(1)

Flask server running...
Open http://localhost:5000


127.0.0.1 - - [03/Sep/2026 16:40:37] "GET / HTTP/1.1" 200 -
